# 03 · Neural networks: residual vs direct learning

Tests the method of Qiao & Wan (2024) on our data: a feed-forward network that learns the hedge ratio directly, against one that learns only the correction to the Black-Scholes delta. Both are compared with the baselines from notebook 02, on the calm validation year (2019) and the crisis test year (2020).

Networks are trained on 2016-18 and stopped early on 2019. Each is trained with several random seeds, because a single run of a network says little about how reliable the number is.

Run notebook 02 first (it saves the Hull-White and LightGBM predictions this notebook reads). A full run trains 15 networks and takes about an hour on a laptop CPU; set `MF703_QUICK=1` for a two-minute smoke test.

In [ ]:
import os, sys, time, pandas as pd, numpy as np, matplotlib.pyplot as plt, torch
sys.path.insert(0, "..")
from config import *
from src import models, evaluate, store
from src.schema import FEATURES

SAMPLE = os.environ.get("MF703_SAMPLE", "main")
QUICK = os.environ.get("MF703_QUICK", "0") == "1"       # tiny run to check the notebook end to end
N_SEEDS = 2 if QUICK else 5
P = sample_paths(SAMPLE)
OUT = P["tables"] / ("neural_quick" if QUICK else "neural"); OUT.mkdir(parents=True, exist_ok=True)
P["figures"].mkdir(parents=True, exist_ok=True)
pd.set_option("display.float_format", "{:.3f}".format)
torch.set_num_threads(os.cpu_count())
print(SAMPLE, "| seeds:", N_SEEDS, "| quick:", QUICK)

In [ ]:
pairs = store.load_pairs(P["pairs"])
pairs = store.load_predictions(pairs, ["hw", "gbm"], P["predictions"])      # from notebook 02
train, val = pairs[pairs.split == "train"], pairs[pairs.split == "val"]
if QUICK:
    train = train.sample(200_000, random_state=0)
print(f"{len(pairs):,} pairs; train {len(train):,}, validation {len(val):,}, test {(pairs.split == 'test').sum():,}")

## The three networks

| Name | Learns | Inputs | Notes |
|---|---|---|---|
| `nn_direct` | the hedge ratio itself | the paper's seven: maturity, delta, moneyness, implied vol, theta, vega, gamma | Qiao & Wan's direct model |
| `nn_resid` | the correction to the Black-Scholes delta | the same seven | Qiao & Wan's residual model, the method under test |
| `nn_resid_hw` | the bracket in Hull-White's formula: correction = g(maturity, delta) x vega / (S sqrt(T)) | maturity and delta only | ours; nests Hull-White, sees no volatility level |

All three use the paper's architecture (three hidden layers of 128 sigmoid units, Adam, learning rate 1e-4, batch 1024). The third was added after the first two failed on 2020; see the note on test-set use at the end.

In [ ]:
SPECS = {
    "nn_direct":   dict(residual=False, features=FEATURES),
    "nn_resid":    dict(residual=True,  features=FEATURES),
    "nn_resid_hw": dict(residual=True,  features=["T", "delta"], output_scale="hw"),
}
FIT = dict(max_epochs=2, patience=2) if QUICK else dict(max_epochs=60, patience=8)

## Train, five seeds each

For every seed the gain ratios are recorded. The model's saved prediction is the average of its seeds' hedge ratios (a small ensemble), which is what later notebooks use.

In [ ]:
test = pairs[pairs.split == "test"]
SUBSETS = {"val": pairs.index[pairs.split == "val"], "test": test.index,
           "crisis": pairs.index[evaluate.crisis_mask(pairs)],
           **{f"test_q{q}": test.index[test["date"].dt.quarter == q] for q in (1, 2, 3, 4)}}

def gains(pred: pd.Series) -> dict:
    """Gain ratio vs Black-Scholes on each subset, calls and puts separately."""
    out = {}
    for tag, idx in SUBSETS.items():
        d = pairs.loc[idx, ["type", "dV", "dS", "delta"]]
        e_m = (d["dV"].astype("float64") - pred.loc[idx].astype("float64") * d["dS"].astype("float64")) ** 2
        e_b = (d["dV"].astype("float64") - d["delta"].astype("float64") * d["dS"].astype("float64")) ** 2
        for t in ("C", "P"):
            k = d["type"] == t
            out[f"{tag}_{t}"] = 1 - e_m[k].sum() / e_b[k].sum()
    return out

runs, histories = [], {}
for name, spec in SPECS.items():
    mean_pred = np.zeros(len(pairs), dtype="float64")
    for seed in range(N_SEEDS):
        t0 = time.time()
        m = models.NeuralHedger(seed=seed, label=name, **spec, **FIT).fit(train, val=val)
        pred = pd.Series(np.asarray(m.predict(pairs), dtype="float64"), index=pairs.index)
        mean_pred += pred.to_numpy() / N_SEEDS
        runs.append({"model": name, "seed": seed, "epochs": len(m.history_), **gains(pred)})
        histories[(name, seed)] = m.history_
        print(f"{name} seed {seed}: {len(m.history_)} epochs, {time.time() - t0:.0f}s, "
              f"2019 {runs[-1]['val_C']:.3f}/{runs[-1]['val_P']:.3f}  2020 {runs[-1]['test_C']:.3f}/{runs[-1]['test_P']:.3f}", flush=True)
    pairs[f"delta_{name}"] = mean_pred.astype("float32")
runs = pd.DataFrame(runs)
runs.to_csv(OUT / "seed_runs.csv", index=False)

## How much do the seeds disagree?

Mean, standard deviation, minimum and maximum of the gain ratio across seeds. Hull-White and LightGBM are shown for reference (they have no seed variation worth speaking of).

In [ ]:
cols = ["val_C", "val_P", "test_C", "test_P", "crisis_C", "crisis_P"]
g = runs.groupby("model")[cols]
seed_summary = pd.concat({"mean": g.mean(), "sd": g.std(), "min": g.min(), "max": g.max()}, axis=1).loc[list(SPECS)]
seed_summary.to_csv(OUT / "seed_summary.csv")
ref = pd.DataFrame({m: gains(pairs[f"delta_{m}"]) for m in ("hw", "gbm")}).T[cols]
display(ref)
# one readable cell per model and column: mean +/- sd [min, max] over the seeds
pd.DataFrame({c: [f"{g.mean().loc[m, c]:.3f} +/- {g.std().loc[m, c]:.3f}  [{g.min().loc[m, c]:.3f}, {g.max().loc[m, c]:.3f}]" for m in SPECS]
              for c in cols}, index=list(SPECS))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=False)
order = ["hw", "gbm"] + list(SPECS)
for ax, col, title in zip(axes, ["val_C", "val_P", "test_C", "test_P"], ["2019 calls", "2019 puts", "2020 calls", "2020 puts"]):
    for i, m in enumerate(order):
        if m in SPECS:
            y = runs.loc[runs.model == m, col]
            ax.scatter([i] * len(y), y, s=22, alpha=0.8)
        else:
            ax.scatter([i], [ref.loc[m, col]], s=60, marker="D")
    ax.axhline(0, color="grey", lw=0.8); ax.axhline(ref.loc["hw", col], color="C0", lw=0.8, ls="--")
    lo = min(-0.05, max(-0.35, runs[col].min() - 0.03))
    ax.set_ylim(lo, max(0.3, runs[col].max() + 0.03)); ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=35, ha="right")
    ax.set_title(title)
    clipped = runs.loc[runs[col] < lo]
    for _, r in clipped.iterrows():
        ax.annotate(f"{r[col]:.2f}", (order.index(r.model), lo), textcoords="offset points", xytext=(6, 4), fontsize=7)
axes[0].set_ylabel("gain ratio vs Black-Scholes")
fig.suptitle("Each dot is one seed; diamonds are the baselines; dashed line = Hull-White", y=1.03)
plt.tight_layout(); plt.savefig(P["figures"] / "neural_seeds.png", dpi=150, bbox_inches="tight")

## Training curves (seed 0)

Validation loss by training pass, against the Black-Scholes and Hull-White validation loss.

In [ ]:
e = lambda col: float(((val["dV"].astype("float64") - val[col].astype("float64") * val["dS"].astype("float64")) ** 2).mean())
fig, ax = plt.subplots(figsize=(7, 3.6))
for name in SPECS:
    h = pd.DataFrame(histories[(name, 0)], columns=["epoch", "train", "val"])
    ax.plot(h["epoch"], h["val"], marker="o", ms=3, label=name)
ax.axhline(e("delta"), color="grey", ls=":", label="Black-Scholes"); ax.axhline(e("delta_hw"), color="C0", ls="--", label="Hull-White")
ax.set_xlabel("training pass"); ax.set_ylabel("validation loss (2019)"); ax.set_ylim(top=e("delta") * 1.15); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(P["figures"] / "neural_training_curves.png", dpi=150)

## All models, seed-averaged networks

The standard tables, with each network represented by the average of its seeds.

In [ ]:
MODELS = ["hw", "gbm"] + list(SPECS)
tables = evaluate.standard_report(pairs, MODELS, OUT)
tables["by_split_type"]

In [ ]:
tables["by_year_type"]

In [ ]:
tables["crisis"]

In [ ]:
by_quarter = evaluate.gain_table(test := pairs[pairs.split == "test"], MODELS, [test["date"].dt.quarter.rename("quarter"), "type"])
by_quarter.to_csv(OUT / "gain_test_by_quarter.csv"); by_quarter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, t in zip(axes, ("C", "P")):
    evaluate.plot_gain_by_bucket(tables["by_delta_bucket"].loc[t], MODELS, f"{'Calls' if t == 'C' else 'Puts'}: test year 2020", ax)
    ax.set_ylim(-0.6, 0.4)
plt.tight_layout(); plt.savefig(P["figures"] / "neural_gain_by_delta.png", dpi=150)

## Why 2020 breaks the networks

Two diagnostics.

**Input drift.** For each of the paper's seven inputs, how far the year's average sits from the training average, in training standard deviations, and what share of values fall outside the range seen in training.

In [ ]:
tr_stats = train[FEATURES].astype("float64").agg(["mean", "std"])
lo, hi = train[FEATURES].quantile(0.001), train[FEATURES].quantile(0.999)
drift = {}
for y in (2019, 2020):
    q = pairs.loc[pairs.year == y, FEATURES].astype("float64")
    drift[(y, "mean, in training sd")] = (q.mean() - tr_stats.loc["mean"]) / tr_stats.loc["std"]
    drift[(y, "share outside training range")] = ((q < lo) | (q > hi)).mean()
drift = pd.DataFrame(drift); drift.to_csv(OUT / "input_drift.csv"); drift

**Correction applied against correction needed.** The hedging-error-weighted average of the needed correction (realised hedge ratio minus Black-Scholes delta) and of the correction each model applied. A model that applies too little leaves gain on the table; one that applies the wrong sign or far too much does worse than Black-Scholes.

In [ ]:
def avg_correction(d):
    w = d["dS"].astype("float64") ** 2
    row = {"needed": ((d["dV"].astype("float64") / d["dS"].astype("float64") - d["delta"].astype("float64")) * w).sum() / w.sum()}
    for m in MODELS:
        row[m] = ((d[f"delta_{m}"].astype("float64") - d["delta"].astype("float64")) * w).sum() / w.sum()
    return pd.Series(row)
corr = pairs[pairs.year >= 2019].groupby(["year", "type"], observed=True).apply(avg_correction)
corr.to_csv(OUT / "correction_applied_vs_needed.csv"); corr

## Screening of other network variants (single seed)

Before settling on the three models above, nine variants were trained once each to see what, if anything, makes a network robust in 2020: inputs divided by the index level ("scale-free"), fewer inputs, clipped inputs, and Hull-White scaling of the output. They were run from a script, not in this notebook; the table is loaded from the saved results. One seed each, so differences of a few hundredths mean little.

In [ ]:
f = TABLES_DIR / "main" / "neural" / "screening_single_seed.csv"
pd.read_csv(f).set_index("model") if f.exists() else "screening table not found"

## Save predictions

In [ ]:
if not QUICK:
    print(store.save_predictions(pairs, list(SPECS), P["predictions"]))

## Findings

*(to be written from the executed results)*